In [1]:
import os
import json
import numpy as np
from pathlib import Path

os.environ['MPLCONFIGDIR'] = f"/tmp/matplotlib_{os.environ.get('USER','user')}"

USER          = os.environ.get('USER', 'sigursteinsson1')
FINAL_DIR     = Path(f"/p/project1/training2600/{USER}/Final_Project")
PROCESSED_DIR = FINAL_DIR / "processed_data"
RESULTS_DIR   = FINAL_DIR / "results"
MODELS_DIR    = FINAL_DIR / "models"
CHECKPOINTS_DIR = FINAL_DIR / "checkpoints"

RESULTS_DIR.mkdir(parents=True, exist_ok=True)
CHECKPOINTS_DIR.mkdir(parents=True, exist_ok=True)

# Read surviving classes from folder structure
surviving_class_names = sorted([
    d.name for d in (PROCESSED_DIR / "train").iterdir() if d.is_dir()
])
NUM_CLASSES = len(surviving_class_names)

print(f"Classes ({NUM_CLASSES}):")
for i, name in enumerate(surviving_class_names):
    print(f"  {i}: {name}")
print(f"\nProcessed data: {PROCESSED_DIR}")
print(f"Checkpoints:    {CHECKPOINTS_DIR}")

Classes (8):
  0: broad_leaved_forest
  1: coniferous_forest
  2: mixed_forest
  3: moors_and_heathland
  4: peat_bogs
  5: transitional_woodland_shrub
  6: water_bodies
  7: water_courses

Processed data: /p/project1/training2600/sigursteinsson1/Final_Project/processed_data
Checkpoints:    /p/project1/training2600/sigursteinsson1/Final_Project/checkpoints


In [2]:
import torch
from torch.utils.data import Dataset, DataLoader
import numpy as np
from pathlib import Path

class CorineDataset(Dataset): # we need this standardised dataset to be able to load all the data one sample at a time
    def __init__(self, root_dir, split, class_names): # builds 2 lists, a file path list as well as a list of integer labels
        self.samples = []
        self.labels  = []
        self.class_to_idx = {name: i for i, name in enumerate(class_names)}
        
        split_dir = Path(root_dir) / split
        for cls_name in class_names:
            cls_dir = split_dir / cls_name
            if not cls_dir.exists():
                continue
            for fpath in cls_dir.glob("*.npy"):
                self.samples.append(fpath)
                self.labels.append(self.class_to_idx[cls_name])
        
        print(f"  {split}: {len(self.samples):,} samples, "
              f"{len(set(self.labels))} classes")

    def __len__(self): # total samples
        return len(self.samples)

    def __getitem__(self, idx): # load one sample -> convert to float32 tensor shape (16, 3, 3) -> Returns it paired with its label (lazy loading)
        patch = np.load(self.samples[idx]).astype(np.float32)
        label = self.labels[idx]
        return torch.from_numpy(patch), torch.tensor(label, dtype=torch.long)

print("Building datasets rn")
train_ds = CorineDataset(PROCESSED_DIR, "train", surviving_class_names)
val_ds   = CorineDataset(PROCESSED_DIR, "val",   surviving_class_names)
test_ds  = CorineDataset(PROCESSED_DIR, "test",  surviving_class_names)

Building datasets rn
  train: 19,838 samples, 8 classes
  val: 9,933 samples, 8 classes
  test: 9,929 samples, 8 classes


In [3]:
# weight sampling and data loaders, similar to how we did it for the baseline MLP and RF, but this time we use distribution chances for every sample, so a water_courses patch with only 136 samples gets a far higher probability than a coniferous_forest path with 8.888.
# the result is that the training batch sees a roughly equal mix of all ~8 of our classes even though our underlying dataset is actually heavily skewed. I used 1e-6 to deal with the divide by 0 problem.

from torch.utils.data import WeightedRandomSampler

# we need to compute per-sample weights to handle the class imbalance in our dataset

label_counts = torch.zeros(NUM_CLASSES)
for lbl in train_ds.labels:
    label_counts[lbl] += 1

class_weights  = 1.0 / (label_counts + 1e-6)
sample_weights = torch.tensor(
    [class_weights[lbl] for lbl in train_ds.labels]
)

sampler = WeightedRandomSampler(
    weights     = sample_weights,
    num_samples = len(train_ds),
    replacement = True,
)

# dataloader set up in line with our given singular compute node with 4 GPU's

BATCH_SIZE  = 128
NUM_WORKERS = 4

train_loader = DataLoader(
    train_ds, batch_size=BATCH_SIZE,
    sampler=sampler, num_workers=NUM_WORKERS, pin_memory=True # pin memory set to True allows for the storage of tensors in a specialized memory region, thus increasing CPU to GPU data transfer
)
val_loader = DataLoader(
    val_ds, batch_size=BATCH_SIZE,
    shuffle=False, num_workers=NUM_WORKERS, pin_memory=True
)
test_loader = DataLoader(
    test_ds, batch_size=BATCH_SIZE,
    shuffle=False, num_workers=NUM_WORKERS, pin_memory=True
)


# IMPORTANT TO NOTE:
# only the training data use the sampler, we want the model to be able to learn and thus we used the balancer to help with that, but we want the val/test set to reflect real class distribution of our data, not an artifically balanced one

print(f"Batch size:     {BATCH_SIZE}")
print(f"Train batches:  {len(train_loader)}") # 155 * 128 = 19.840
print(f"Val batches:    {len(val_loader)}")   # 78 * 128  = 9.933
print(f"Test batches:   {len(test_loader)}")  # 78 * 128  = 9.933

Batch size:     128
Train batches:  155
Val batches:    78
Test batches:   78


In [22]:
import torch.nn as nn
import terratorch
import logging
from terratorch.models import EncoderDecoderFactory

# Check terratorch version to know which API to use
import importlib.metadata
print(f"TerraTorch version: {importlib.metadata.version('terratorch')}")

try:
    factory = EncoderDecoderFactory()

    logging.getLogger("terratorch").setLevel(logging.ERROR)
    
    model = factory.build_model(
        task            = "classification",
        backbone        = "prithvi_eo_v2_300",
        backbone_kwargs = {
            "pretrained" : True,
            "num_frames" : 4,
            "in_chans"   : 6,  # we only have 4 channels we use, but we need to match the format Prithvi expects
            "img_size"   : 32, # original tensor shape is 3x3 but instead of bilinear upsampling to 32x32, we want to use Prithvi as a true end-to-end fine-tuned model rather than a bad feature extractor, mimicking the labs.
        },
        decoder         = "IdentityDecoder",
        decoder_kwargs  = {},
        head_kwargs     = {"num_classes": NUM_CLASSES},
    )
    print(f"Model built successfully")
    print(f"Total parameters:     {sum(p.numel() for p in model.parameters()):,}")
    print(f"Trainable parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")
    
except Exception as e:
    print(f"Build failed: {e}")

TerraTorch version: 1.2.6


2026-04-27 12:25:37,020 - INFO - HTTP Request: HEAD https://huggingface.co/ibm-nasa-geospatial/Prithvi-EO-2.0-300M/resolve/main/Prithvi_EO_V2_300M.pt "HTTP/1.1 302 Found"
2026-04-27 12:25:37,469 - INFO - Loaded weights for HLSBands.BLUE in position 0 of patch embed
2026-04-27 12:25:37,470 - INFO - Loaded weights for HLSBands.GREEN in position 1 of patch embed
2026-04-27 12:25:37,471 - INFO - Loaded weights for HLSBands.RED in position 2 of patch embed
2026-04-27 12:25:37,472 - INFO - Loaded weights for HLSBands.NIR_NARROW in position 3 of patch embed
2026-04-27 12:25:37,509 - INFO - Loaded weights for HLSBands.SWIR_1 in position 4 of patch embed
2026-04-27 12:25:37,511 - INFO - Loaded weights for HLSBands.SWIR_2 in position 5 of patch embed


Model built successfully
Total parameters:     303,919,112
Trainable parameters: 303,919,112


In [27]:
# We want to check a sample patch to see the channel oreding for Prithvi, should be March -> May -> Oct
import numpy as np
sample = np.load(list((PROCESSED_DIR / "train" / "coniferous_forest").glob("*.npy"))[0])
print(f"Patch shape: {sample.shape}")
print(f"Channel 0 (B02 March) stats: min={sample[0].min():.3f} max={sample[0].max():.3f}")
print(f"Channel 4 (B02 May)   stats: min={sample[4].min():.3f} max={sample[4].max():.3f}")
print(f"Channel 8 (B02 Aug)   stats: min={sample[8].min():.3f} max={sample[8].max():.3f}")
print(f"Channel 12 (B02 Oct)  stats: min={sample[12].min():.3f} max={sample[12].max():.3f}")

Patch shape: (16, 3, 3)
Channel 0 (B02 March) stats: min=0.059 max=0.182
Channel 4 (B02 May)   stats: min=0.079 max=0.107
Channel 8 (B02 Aug)   stats: min=0.011 max=0.028
Channel 12 (B02 Oct)  stats: min=0.010 max=0.087


In [ ]:
# Prithvi training loop

# NOTE: we had to match Prithvi's weight shape, it expects 6 channels as inputs but we only have 4
# To fix this we zero padded our 4 bands to 6 by adding two empty channels for the missing SWIR bands.
# Raw data: (B, 16, 3, 3)
# Reshape it: (B, 4, 4, 3, 3)
# Merge Dims: (B*6*4, 1, 3, 3)
# Bilinear upsamling of spatial dims: (B*6*4, 1, 32, 32)
# Pritvhi ready reshaping: (B, 6, 4, 32, 32)

import torch.optim as optim
from torch.optim.lr_scheduler import CosineAnnealingLR
from sklearn.metrics import accuracy_score, f1_score

DEVICE   = torch.device("cuda" if torch.cuda.is_available() else "cpu") # GPU with a CPU fallback
N_EPOCHS = 20 # modify this as needed, 20 is the goal
LR       = 1e-4 # Division by 0 solution

print(f"Training on: {DEVICE}")
if DEVICE.type == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

model = model.to(DEVICE)

# Freeze backbone for first 5 epochs then only train the classification head
def set_backbone_grad(model, requires_grad):
    for name, param in model.named_parameters():
        if "head" not in name:
            param.requires_grad = requires_grad

set_backbone_grad(model, False)
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"\nPhase 1 (epochs 1-5): backbone frozen, trainable params: {trainable:,}")

optimizer = optim.AdamW(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=LR, weight_decay=1e-4
)

# By using CosineAnnealingLR we can gradually reduce the learning rate following a consine curve over our 20 epochs. This prevents the model from oscilliating around the minimum at the end of the training
scheduler = CosineAnnealingLR(optimizer, T_max=N_EPOCHS)
criterion = torch.nn.CrossEntropyLoss()

best_val_f1 = 0.0
best_ckpt   = CHECKPOINTS_DIR / "prithvi_best.pt"
history     = []

for epoch in range(1, N_EPOCHS + 1):

    # Unfreeze backbone after epoch 5
    if epoch == 6:
        set_backbone_grad(model, True)
        optimizer = optim.AdamW(
            model.parameters(), lr=LR * 0.1, weight_decay=1e-4
        )
        trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
        print(f"\nPhase 2 (epoch 6+): backbone unfrozen, trainable params: {trainable:,}")

    # Train
    model.train()
    train_loss = 0.0
    for X, y in train_loader:
        X, y = X.to(DEVICE), y.to(DEVICE)
        # Reshape (B, 16, 3, 3) → (B, 4bands, 4times, 3, 3)
        B, _, H, W = X.shape
        X = X.reshape(B, 4, 4, H, W)          # (B, 4bands, 4times, 3, 3)
        
        # Zero-pad bands from 4 to 6 (add empty SWIR_1 and SWIR_2 channels)
        pad = torch.zeros(B, 2, 4, H, W, device=DEVICE)
        X = torch.cat([X, pad], dim=1)         # (B, 6bands, 4times, 3, 3)
        
        # Upsample spatial dims 3→32
        X = X.reshape(B * 6 * 4, 1, H, W)
        X = torch.nn.functional.interpolate(X, size=(32, 32), mode="bilinear", align_corners=False)
        X = X.reshape(B, 6, 4, 32, 32)        # (B, 6bands, 4times, 32, 32)
        optimizer.zero_grad()
        out    = model(X)
        logits = out.output if hasattr(out, "output") else out
        loss   = criterion(logits, y)
        loss.backward()
        optimizer.step()
        train_loss += loss.item()

    # Validate
    model.eval()
    all_preds, all_labels = [], []
    val_loss = 0.0
    with torch.no_grad():
        for X, y in val_loader:
            X, y = X.to(DEVICE), y.to(DEVICE)
            # Reshape (B, 16, 3, 3) → (B, 4bands, 4times, 3, 3)
            B, _, H, W = X.shape
            X = X.reshape(B, 4, 4, H, W)          # (B, 4bands, 4times, 3, 3)
            
            # Zero-pad bands from 4 to 6 (add empty SWIR_1 and SWIR_2 channels)
            pad = torch.zeros(B, 2, 4, H, W, device=DEVICE)
            X = torch.cat([X, pad], dim=1)         # (B, 6bands, 4times, 3, 3)
            
            # Upsample spatial dims 3→32
            X = X.reshape(B * 6 * 4, 1, H, W)
            X = torch.nn.functional.interpolate(X, size=(32, 32), mode="bilinear", align_corners=False)
            X = X.reshape(B, 6, 4, 32, 32)        # (B, 6bands, 4times, 32, 32)
            out    = model(X)
            logits = out.output if hasattr(out, "output") else out
            val_loss += criterion(logits, y).item()
            all_preds.extend(logits.argmax(dim=1).cpu().numpy())
            all_labels.extend(y.cpu().numpy())

    val_oa = accuracy_score(all_labels, all_preds)
    val_f1 = f1_score(all_labels, all_preds, average="macro", zero_division=0)

    history.append({
        "epoch"       : epoch,
        "train_loss"  : train_loss / len(train_loader),
        "val_loss"    : val_loss   / len(val_loader),
        "val_oa"      : val_oa,
        "val_macro_f1": val_f1,
    })

    print(f"Epoch {epoch:2d}/{N_EPOCHS} | "
          f"train_loss: {train_loss/len(train_loader):.4f} | "
          f"val_loss: {val_loss/len(val_loader):.4f} | "
          f"val_OA: {val_oa:.4f} | val_F1: {val_f1:.4f}")

    if val_f1 > best_val_f1:
        best_val_f1 = val_f1
        torch.save(model.state_dict(), best_ckpt)
        print(f" Best checkpoint saved (val F1={val_f1:.4f})")

    scheduler.step()

print(f"\nTraining complete. Best val F1: {best_val_f1:.4f}")

# Save training history
import json
with open(RESULTS_DIR / "prithvi_training_history.json", "w") as f:
    json.dump(history, f, indent=2)
print(f"History saved.")

Training on: cuda
GPU: Quadro RTX 8000
VRAM: 47.6 GB

Phase 1 (epochs 1-5): backbone frozen, trainable params: 32,776
Epoch  1/20 | train_loss: 0.1364 | val_loss: 3.3149 | val_OA: 0.3404 | val_F1: 0.1379
 Best checkpoint saved (val F1=0.1379)
Epoch  2/20 | train_loss: 0.1234 | val_loss: 3.4409 | val_OA: 0.3410 | val_F1: 0.1376
Epoch  3/20 | train_loss: 0.1161 | val_loss: 3.5403 | val_OA: 0.3345 | val_F1: 0.1391
 Best checkpoint saved (val F1=0.1391)


In [29]:
import json
import numpy as np
from sklearn.metrics import (
    accuracy_score, f1_score, classification_report, confusion_matrix
)

# Load best checkpoint
print(f"Loading best checkpoint from: {best_ckpt}")
state = torch.load(best_ckpt, map_location=DEVICE, weights_only=True)
model.load_state_dict(state)
model.eval()
print(f"Checkpoint loaded. Best val F1 was: {best_val_f1:.4f}")

# Run inference on the full test set
print(f"\nRunning inference on test set ({len(test_ds):,} patches)...")
all_preds, all_labels = [], []

with torch.no_grad():
    for X, y in test_loader:
        X, y = X.to(DEVICE), y.to(DEVICE)

        # Same preprocessing as training: (B, 16, 3, 3) -> (B, 6, 4, 32, 32), otherwise numbers are meaningless
        B, _, H, W = X.shape
        X = X.reshape(B, 4, 4, H, W)
        pad = torch.zeros(B, 2, 4, H, W, device=DEVICE)
        X = torch.cat([X, pad], dim=1)
        X = X.reshape(B * 6 * 4, 1, H, W)
        X = torch.nn.functional.interpolate(
            X, size=(32, 32), mode="bilinear", align_corners=False
        )
        X = X.reshape(B, 6, 4, 32, 32)

        out    = model(X)
        logits = out.output if hasattr(out, "output") else out
        all_preds.extend(logits.argmax(dim=1).cpu().numpy())
        all_labels.extend(y.cpu().numpy())

all_preds  = np.array(all_preds)
all_labels = np.array(all_labels)

# Compute aggregate metrics
test_oa = accuracy_score(all_labels, all_preds)
test_f1 = f1_score(all_labels, all_preds, average="macro", zero_division=0) # water_courses is too small thus we need the zero_division to remove the warnings for a non predicted class

print(f"\n{'='*60}")
print(f"PRITHVI-EO-2.0 TEST SET RESULTS")
print(f"{'='*60}")
print(f"Test Overall Accuracy: {test_oa:.4f}")
print(f"Test Macro F1:         {test_f1:.4f}")
print(f"{'='*60}")

# Per-class breakdown
print("\nPer-class classification report:")
report_str = classification_report(
    all_labels, all_preds,
    target_names=surviving_class_names,
    digits=4,
    zero_division=0,
)
print(report_str)

# Also get the report as a dict so we can save structured data
report_dict = classification_report(
    all_labels, all_preds,
    target_names=surviving_class_names,
    digits=4,
    zero_division=0,
    output_dict=True,
)

# Confusion matrix
cm = confusion_matrix(all_labels, all_preds, labels=list(range(NUM_CLASSES)))
print("\nConfusion matrix (rows = true, cols = predicted):")
print("Row/Col order:")
for i, name in enumerate(surviving_class_names):
    print(f"  {i}: {name}")
print()
print(cm)

# Save predictions for later figure generation
np.save(RESULTS_DIR / "prithvi_test_preds.npy",  all_preds)
np.save(RESULTS_DIR / "prithvi_test_labels.npy", all_labels)
np.save(RESULTS_DIR / "prithvi_confusion_matrix.npy", cm)
print(f"\nPredictions saved to: {RESULTS_DIR}")

# Update all_results.json with Prithvi results so we dont override RF and MLP
results_path = RESULTS_DIR / "all_results.json"
if results_path.exists():
    with open(results_path) as f:
        all_results = json.load(f)
else:
    all_results = {}

all_results["prithvi_eo_v2_300"] = {
    "test_oa":       float(test_oa),
    "test_macro_f1": float(test_f1),
    "best_val_f1":   float(best_val_f1),
    "n_epochs":      N_EPOCHS,
    "lr_phase1":     LR,
    "lr_phase2":     LR * 0.1,
    "batch_size":    BATCH_SIZE,
    "per_class":     {
        name: {
            "precision": report_dict[name]["precision"],
            "recall":    report_dict[name]["recall"],
            "f1":        report_dict[name]["f1-score"],
            "support":   report_dict[name]["support"],
        }
        for name in surviving_class_names
    },
}

with open(results_path, "w") as f:
    json.dump(all_results, f, indent=2)

print(f"Updated: {results_path}")

# Quick comparison print
print(f"\n{'─'*60}")
print("Quick comparison (test macro F1):")
print(f"{'─'*60}")
for model_name, res in all_results.items():
    f1 = res.get("test_macro_f1", res.get("macro_f1", "N/A"))
    oa = res.get("test_oa", res.get("overall_accuracy", "N/A"))
    if isinstance(f1, float):
        print(f"  {model_name:<30} OA: {oa:.4f}  Macro F1: {f1:.4f}")
    else:
        print(f"  {model_name:<30} OA: {oa}  Macro F1: {f1}")

Loading best checkpoint from: /p/project1/training2600/sigursteinsson1/Final_Project/checkpoints/prithvi_best.pt
Checkpoint loaded. Best val F1 was: 0.1451

Running inference on test set (9,929 patches)...

PRITHVI-EO-2.0 TEST SET RESULTS
Test Overall Accuracy: 0.3239
Test Macro F1:         0.1370

Per-class classification report:
                             precision    recall  f1-score   support

        broad_leaved_forest     0.0552    0.0987    0.0708       314
          coniferous_forest     0.5328    0.4425    0.4835      5028
               mixed_forest     0.1055    0.0641    0.0798       873
        moors_and_heathland     0.0469    0.0359    0.0406       251
                  peat_bogs     0.2562    0.3336    0.2899      2458
transitional_woodland_shrub     0.0774    0.0929    0.0844       700
               water_bodies     0.0249    0.0347    0.0290       259
              water_courses     0.0149    0.0217    0.0177        46

                   accuracy                 